# e2000fs_calib_offset_roll_scap_arrival_cme_smooth_tail_nospec_seam — 수집부터 72시간 예보까지

예보 시각 `t0` 하나에 대해 **원자료 수집 → 특징 39개 → 체크포인트 추론 → block-seam 평활화 → `forecast.json`** 을 이 노트북 안에서 끝냅니다.

- **학습은 하지 않습니다.** 보드 행의 기본 모델 `e2000fs_calib_offset_roll_scap_arrival_cme_smooth_tail_nospec`(smooth-tail A900 + spectral 항 없는 S-cap continuation, 입력 39개)을 **2026-10-31 23Z까지의 label로 다시 학습한 serving 체크포인트** `research/e2000fs_calib_offset_roll_scap_arrival_cme_smooth_tail_nospec_seam/weights/L/s{42,43,44}.npz`(로컬 전용, git 제외)를 불러와 추론만 합니다.
  - 학습 origin 2022-10-01T00Z ~ 2026-10-28T23Z(마지막 target 10-31T23Z), raw 입력도 모두 2026-11-01T00Z 이전 것만 씁니다(`~/sunrun-seam-live/build_live_table.py`, `train_live.py`).
  - 보드 기간(F1/F2/F3)보다 나중 자료로 학습했으므로 이 체크포인트는 리더보드로 채점할 수 없습니다.
- **seam 평활화 (보드 행 v2, xf8):** 3개 블록(lead 1–24, 25–48, 49–72)의 이음매 h24|25, h48|49에서 국소 기울기를 넘는 계단을 양쪽에서 절반씩 빼고 8개 lead에 걸쳐 선형으로 줄입니다. seed 예보마다 적용한 뒤 세 seed를 등가중 평균합니다. k = 8은 보드 행 v2와 같습니다(사용자 요청으로 리더보드 xf1..xf24 sweep에서 정한 값, 학습과 무관한 고정 후처리).
- **결측:** 학습 표와 같이 t0 앞 168개 origin을 다시 만들어 forward fill합니다. 아직 공개되지 않은 가장 최근 ACE 칸은 SWPC 실시간 1분/5분 값의 평균으로 채우고, 실시간 값이 없으면 forward fill로 남습니다.
- **다른 모듈을 import하지 않습니다.** 수집·특징·추론·평활화 로직은 `benchlab-submission/pipeline`, `research/_module/public_events.py`, `research/_module/{gbm,continuation,calibration}.py`, 보드 행 `experiment.py`의 `smooth`에서 그대로 옮겨 셀에 넣었습니다. 필요한 것은 numpy, pandas, scipy, h5py, xgboost와 체크포인트 파일뿐입니다.
- 모든 입력은 `t0` 이전에 공개된 것만 씁니다.

| 단계 | 내용 |
|---|---|
| 0 | 설정: `T0`, 체크포인트 경로, seam taper, 특징 목록 |
| 1 | 수집: ACE 월별 1h + SWPC 실시간 1m/5m, DONKI WSA-ENLIL, GOES-18 SUVI 1장, DONKI CME notifications |
| 2 | 전처리: ACE·SUVI·ENLIL 특징 31개 + DONKI 도착 시각(T)·CME 운동학(K) 특징 8개 |
| 3 | 추론: seed별 3블록(A900 booster + continuation stage) + lead별 offset |
| 3.5 | seam 평활화: seed마다 xf8, 3 seed 평균 |
| 4 | 예보 기록: 72개 값 확인, `forecast.json` |
| 5 | 검증: 추론·평활화 코드가 보드 행 F3 예측과 새 체크포인트의 학습 시 예측을 그대로 재현하는지 확인 |

실행(GBM은 CPU 전용): `CUDA_VISIBLE_DEVICES= jupyter nbconvert --to notebook --execute --inplace end_to_end.ipynb`. 환경 변수 `T0=20261118T060000Z`로 예보 시각을 바꿉니다.

## 0. 설정

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""            # GBM은 CPU만 사용
THREADS = 4
for _key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_key] = str(THREADS)                # OMP_NUM_THREADS=1이면 XGBoost가 단일 thread가 됨

import json, re, tempfile, time, urllib.error, urllib.parse, urllib.request, warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from types import SimpleNamespace

import h5py
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import ndimage

REPO = Path("/home/t-lab01/sunrun")
EXPERIMENT = "e2000fs_calib_offset_roll_scap_arrival_cme_smooth_tail_nospec_seam"
BASE_ROW = "e2000fs_calib_offset_roll_scap_arrival_cme_smooth_tail_nospec"   # seam 평활화 전 모델
LIVE = Path.home() / "sunrun-seam-live"
PERIOD, SEEDS = "L", (42, 43, 44)                   # L = label 2026-10-31T23Z까지로 다시 학습한 체크포인트
CHECKPOINTS = {seed: REPO / "research" / EXPERIMENT / "weights" / PERIOD / f"s{seed}.npz" for seed in SEEDS}
assert all(p.exists() for p in CHECKPOINTS.values()), "체크포인트가 없습니다"
SEAM_TAPER = 8                                      # 보드 행 v2의 xf8

# 예보 시각: 환경 변수 T0(예: 20261116T060000Z)가 없으면 지금 시각의 정시
T0_TEXT = os.environ.get("T0") or pd.Timestamp.now(tz="UTC").floor("h").strftime("%Y%m%dT%H%M%SZ")
T0 = pd.Timestamp(pd.to_datetime(T0_TEXT, format="%Y%m%dT%H%M%SZ", utc=True))
RUN_DIR = LIVE / "forecasts" / T0_TEXT
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 모델 입력 (booster 열 순서 = 부모 상수 20, lead별 8 x 24, 추가 상수 3 + DONKI 8)
PARENT_CONSTANT = ["v_lag0", "ace_speed_kms_last", "ace_density_cm3_last", "ace_density_cm3_m6",
                   "ace_temperature_k_last", "ace_temperature_k_m24", "ace_bt_nt_last", "ace_bt_nt_m6",
                   "ace_by_gsm_nt_m24", "ace_epam_p_channel3_last", "ace_sis_gt10_flux_last",
                   "suvi_t035_top2_area", "suvi_t035_meridian15_equatorial_area",
                   "suvi_t035_meridian30_equatorial_area", "suvi_t055_meridian15_area",
                   "suvi_t055_meridian30_equatorial_area", "suvi_t055_equatorial_area", "img_age_h",
                   "enlil_n_pending", "enlil_kp"]
ADDED_CONSTANT = ["v_lag24", "v_lag48", "v_lag72"]
T_FEATURES = ["pub_enlil_next_arr_h", "pub_enlil_since_arr_h", "pub_enlil_glancing"]
K_FEATURES = ["pub_cme_next_speed_kms", "pub_cme_next_half_angle_deg", "pub_cme_next_lon_deg",
              "pub_cme_next_lat_deg", "pub_cme_last_speed_kms"]
DONKI_FEATURES = T_FEATURES + K_FEATURES
CONSTANT = PARENT_CONSTANT + ADDED_CONSTANT + DONKI_FEATURES
VARYING = ["rec26", "rec27", "rec28", "rec_mean", "strip_v350_logmed", "strip_v450_logmed", "strip_v550_dark",
           "strip_v750_logmed"]

H, BLOCK = 72, 24                                   # 예보 lead 72시간, booster 하나가 24 lead
HOUR = pd.Timedelta(hours=1)
# NASA 월별 ACE 파일은 매시 약 :10에 갱신됩니다. 라벨 L 행은 L + 70분 <= t0일 때만 공개된 것으로 봅니다.
ACE_PUBLISH_DELAY = pd.Timedelta(minutes=70)
ACE_PRODUCTS = ("swepam", "mag", "epam", "sis")
# SWPC 실시간 목록: 표시 시각 s의 행은 s + 지연 <= t0일 때 공개된 것으로 봅니다.
ACE_1M_LAG = {"swepam": pd.Timedelta(minutes=4), "mag": pd.Timedelta(minutes=4),
              "epam": pd.Timedelta(minutes=10), "sis": pd.Timedelta(minutes=10)}
LOOKBACK_ORIGINS = 168                              # 학습의 forward fill을 재현하려고 t0 앞 168개 origin을 다시 만듦
FALLBACK_KMS = 420.0                                # 속도 관측이 하나도 없을 때의 마지막 대체값


def bin_hour(t):
    """학습 규칙: 시각 t는 칸 floor(t, 1h) + 1h에 들어가고, origin O는 칸 <= O만 씁니다."""
    return pd.DatetimeIndex(pd.to_datetime(t, utc=True)).floor("h") + HOUR


print("t0        :", T0)
print("run folder:", RUN_DIR)
print("checkpoint:", *[str(p) for p in CHECKPOINTS.values()], sep="\n  ")
print("seam      :", f"xf{SEAM_TAPER}")
print("features  :", len(CONSTANT), "constant +", len(VARYING), "per-lead =", len(CONSTANT) + len(VARYING))

## 1. 수집

네트워크를 쓰는 곳은 이 단계뿐입니다. 각 원천에서 `t0`에 이미 공개돼 있던 것만 남깁니다.

| 원천 | 주소 | 공개 시각 규칙 |
|---|---|---|
| ACE SWEPAM·MAG·EPAM·SIS 1시간 (월별 파일, 이번 달 + 지난달) | `sohoftp.nascom.nasa.gov` | 라벨 `L` 행은 `L + 70분 <= t0` |
| ACE 실시간 1분(SWEPAM·MAG)·5분(EPAM·SIS), 최근 2시간 | `services.swpc.noaa.gov/text/` | 표시 시각 `s` 행은 `s + 4분`(1분 목록) 또는 `s + 10분`(5분 목록) `<= t0` |
| WSA-ENLIL 실행, 최근 9일 | `ccmc.gsfc.nasa.gov/DONKI/WS` | `modelCompletionTime <= t0` |
| GOES-18 SUVI Fe195 L1b 1장 | `noaa-goes18.s3.amazonaws.com` | 프레임과 NetCDF의 S3 `LastModified <= t0` |
| DONKI CME notifications, 최근 30일 | `ccmc.gsfc.nasa.gov/DONKI-API` | `messageIssueTime <= t0` (특징 계산은 본문 발행 시각 `<= origin`으로 한 번 더 자름) |

### 1.1 HTTP

In [ ]:
UA = "benchlab-offset-roll/1.0"


class Deadline:
    """Wall-clock budget for the whole collect layer, so a slow source cannot eat the 15-minute run limit."""

    def __init__(self, seconds: float):
        self.end = time.monotonic() + seconds

    def left(self) -> float:
        return self.end - time.monotonic()


def get(url: str, deadline: Deadline, byte_range: tuple[int, int] | None = None, tries: int = 3,
        timeout: float = 30.0) -> bytes | None:
    """Body of `url`, None on 404. Other failures are retried and then raised (the caller decides what a miss means)."""
    headers = {"User-Agent": UA}
    if byte_range:
        headers["Range"] = f"bytes={byte_range[0]}-{byte_range[1]}"
    failure: Exception | None = None
    for attempt in range(tries):
        if deadline.left() <= 0:
            raise TimeoutError(f"collect deadline reached before {url}")
        try:
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=min(timeout, max(1.0, deadline.left()))) as response:
                return response.read()
        except urllib.error.HTTPError as error:
            if error.code == 404:
                return None
            failure = error
        except OSError as error:
            failure = error
        if attempt < tries - 1:
            time.sleep(min(2.0 * (attempt + 1), max(0.0, deadline.left())))
    raise RuntimeError(f"GET failed {url}: {failure}")

### 1.2 ACE 월별 1시간 파일과 실시간 1분/5분 목록

In [ ]:
ACE_BASE = "https://sohoftp.nascom.nasa.gov/sdb/goes/ace/monthly/"
# SWPC real-time lists of the last two hours: 1-minute SWEPAM/MAG, 5-minute EPAM/SIS (same row layout as the monthly files)
ACE_BASE_1M = "https://services.swpc.noaa.gov/text/"
NAME_1M = {"swepam": "ace-swepam", "mag": "ace-magnetometer", "epam": "ace-epam", "sis": "ace-sis"}


def months_for(t0: pd.Timestamp) -> list[str]:
    """The month of t0 and the one before it: 28 days of history (rec26..28) never reach further back."""
    this = t0.strftime("%Y%m")
    before = (t0.replace(day=1) - pd.Timedelta(days=1)).strftime("%Y%m")
    return [before, this]


def row_label(line: str) -> pd.Timestamp | None:
    parts = line.split()
    if len(parts) < 4 or not parts[0].isdigit():
        return None
    hhmm = int(parts[3])
    return pd.Timestamp(year=int(parts[0]), month=int(parts[1]), day=int(parts[2]), tz="UTC") + pd.Timedelta(
        hours=hhmm // 100, minutes=hhmm % 100)


def as_of(text: str, t0: pd.Timestamp) -> str:
    """Header lines plus the rows already published at t0 (label + ACE_PUBLISH_DELAY <= t0)."""
    kept = []
    for line in text.splitlines():
        label = row_label(line)
        if label is None or label + ACE_PUBLISH_DELAY <= t0:
            kept.append(line)
    return "\n".join(kept) + "\n"


def collect_ace(t0: pd.Timestamp, deadline: Deadline, log: list[str]) -> dict[str, list[str]]:
    out: dict[str, list[str]] = {}
    for product in ACE_PRODUCTS:
        texts = []
        for month in months_for(t0):
            url = f"{ACE_BASE}{month}_ace_{product}_1h.txt"
            try:
                body = get(url, deadline)
            except (RuntimeError, TimeoutError) as error:
                log.append(f"ace {product} {month}: FAILED {error}")
                continue
            if body is None:
                log.append(f"ace {product} {month}: not found")
                continue
            text = as_of(body.decode("ascii", "replace"), t0)
            texts.append(text)
            rows = [row_label(line) for line in text.splitlines()]
            rows = [r for r in rows if r is not None]
            log.append(f"ace {product} {month}: {len(rows)} rows published by t0, newest label {rows[-1] if rows else None}")
        out[product] = texts
    return out


def as_of_1m(text: str, t0: pd.Timestamp, lag: pd.Timedelta) -> str:
    """Header lines plus the rows already on the real-time list at t0 (row stamp + lag <= t0)."""
    kept = []
    for line in text.splitlines():
        stamp = row_label(line)
        if stamp is None or stamp + lag <= t0:
            kept.append(line)
    return "\n".join(kept) + "\n"


def collect_ace_1m(t0: pd.Timestamp, deadline: Deadline, log: list[str]) -> dict[str, str]:
    """The real-time 1m/5m list of each product, restricted to what was published at t0.

    The list only reaches back two hours, so this serves the live run (and a replay within about an hour of t0). A
    product that cannot be fetched is left out and the monthly-file forward fill stays in force for its columns.
    """
    out: dict[str, str] = {}
    for product in ACE_PRODUCTS:
        url = f"{ACE_BASE_1M}{NAME_1M[product]}.txt"
        try:
            body = get(url, deadline)
        except (RuntimeError, TimeoutError) as error:
            log.append(f"ace_1m {product}: FAILED {error}")
            continue
        if body is None:
            log.append(f"ace_1m {product}: not found")
            continue
        text = as_of_1m(body.decode("ascii", "replace"), t0, ACE_1M_LAG[product])
        out[product] = text
        stamps = [s for s in (row_label(line) for line in text.splitlines()) if s is not None]
        log.append(f"ace_1m {product}: {len(stamps)} rows published by t0, newest {stamps[-1] if stamps else None}")
    return out

### 1.3 DONKI WSA-ENLIL 실행

In [ ]:
ENLIL_URL = "https://ccmc.gsfc.nasa.gov/DONKI/WS/get/WSAEnlilSimulations"
ENLIL_LOOKBACK = pd.Timedelta(days=9)           # the feature window is 7 days of completion bins; 2 spare days


def collect_enlil(t0: pd.Timestamp, deadline: Deadline, log: list[str]) -> list[dict]:
    start, end = (t0 - ENLIL_LOOKBACK).strftime("%Y-%m-%d"), t0.strftime("%Y-%m-%d")
    try:
        body = get(f"{ENLIL_URL}?startDate={start}&endDate={end}", deadline, timeout=60.0)
    except (RuntimeError, TimeoutError) as error:
        log.append(f"enlil: FAILED {error}")
        return []
    runs = json.loads(body) if body and body.strip() else []
    kept = []
    for run in runs:
        done = run.get("modelCompletionTime")
        if done and pd.Timestamp(done) <= t0:
            kept.append({k: run.get(k) for k in ("simulationID", "modelCompletionTime", "estimatedShockArrivalTime",
                                                 "kp_90", "kp_135", "kp_180", "isEarthGB")})
    log.append(f"enlil: {len(runs)} simulations {start}..{end}, {len(kept)} completed by t0")
    return kept

### 1.4 SUVI 프레임

학습 때 고른 규칙 그대로입니다. 칸 B의 프레임은 관측 끝이 [B−1h, B) 안에 있고 `obs_end + 2초 <= B`인 것 중에서 식(eclipse) 없는 것, 노출이 가장 긴 것, 가장 늦은 것 순으로 고릅니다. 칸에 쓸 프레임이 없으면 최대 12칸 앞까지 거슬러 올라갑니다.

In [ ]:
S3_BASE = "https://noaa-goes18.s3.amazonaws.com"
CHANNEL = "Fe195"
S3_NS = "{http://s3.amazonaws.com/doc/2006-03-01/}"
SUVI_NAME = re.compile(r"_s(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_e(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_c")
MARGIN = pd.Timedelta(seconds=2)
DUR_TOLERANCE = 0.15
FITS_BLOCK = 2880
MAX_BACK = 12


def suvi_stamp(groups) -> pd.Timestamp:
    year, doy, hh, mm, ss, tenth = (int(x) for x in groups)
    return pd.Timestamp(year=year, month=1, day=1, tz="UTC") + pd.Timedelta(
        days=doy - 1, hours=hh, minutes=mm, seconds=ss, milliseconds=100 * tenth)


def hour_prefix(hour: pd.Timestamp) -> str:
    return f"SUVI-L1b-{CHANNEL}/{hour:%Y}/{hour.dayofyear:03d}/{hour:%H}/"


def list_hour(hour: pd.Timestamp, deadline: Deadline) -> tuple[list[dict], dict[str, pd.Timestamp]]:
    """FITS frames of one S3 hour folder (oldest first) and LastModified of every object in it."""
    frames, modified, token = [], {}, None
    while True:
        query = {"list-type": "2", "prefix": hour_prefix(hour)}
        if token:
            query["continuation-token"] = token
        body = get(f"{S3_BASE}/?{urllib.parse.urlencode(query)}", deadline)
        root = ET.fromstring(body)
        for item in root.findall(f"{S3_NS}Contents"):
            key = item.findtext(f"{S3_NS}Key")
            modified[key] = pd.Timestamp(item.findtext(f"{S3_NS}LastModified"))
            match = SUVI_NAME.search(key)
            if key.endswith(".fits") and match:
                start, end = suvi_stamp(match.groups()[:6]), suvi_stamp(match.groups()[6:])
                frames.append(dict(key=key, obs_end=end, dur=(end - start).total_seconds()))
        if root.findtext(f"{S3_NS}IsTruncated") != "true":
            return sorted(frames, key=lambda f: f["obs_end"]), modified
        token = root.findtext(f"{S3_NS}NextContinuationToken")


def parse_header(raw: bytes) -> tuple[dict, bool]:
    cards = {}
    for offset in range(0, len(raw) - 79, 80):
        card = raw[offset:offset + 80].decode("ascii", "replace")
        if card.startswith("END") and card[3:].strip() == "":
            return cards, True
        if len(card) > 9 and card[8] == "=":
            cards[card[:8].strip()] = card[10:].split(" /")[0].strip().strip("'").strip()
    return cards, False


def read_header(key: str, deadline: Deadline) -> dict:
    for blocks in (5, 12, 60):
        raw = get(f"{S3_BASE}/{key}", deadline, byte_range=(0, blocks * FITS_BLOCK - 1))
        if raw is None:
            raise RuntimeError(f"FITS object vanished: {key}")
        cards, done = parse_header(raw)
        if done:
            break
    else:
        raise ValueError(f"no END card: {key}")

    def number(name):
        try:
            return float(cards[name])
        except (KeyError, ValueError):
            return None
    return dict(exptime=number("EXPTIME"), eclipse=number("ECLIPSE"))


def select_for_bin(slot: pd.Timestamp, t0: pd.Timestamp, deadline: Deadline):
    """(chosen frame, LastModified map) for hour bin `slot`, or (None, map) when nothing was usable at t0."""
    frames, modified = list_hour(slot - HOUR, deadline)
    usable = [f for f in frames if f["obs_end"] + MARGIN <= slot and modified[f["key"]] <= t0
              and modified.get(f["key"][:-5] + ".nc", t0 + HOUR) <= t0]
    if not usable:
        return None, modified
    longest = max(f["dur"] for f in usable)
    scored = []
    for frame in usable:
        if frame["dur"] >= longest - DUR_TOLERANCE:
            header = read_header(frame["key"], deadline)
            exposure = round(header["exptime"], 1) if header["exptime"] is not None else -1
            scored.append(((header["eclipse"] == 0, exposure, frame["obs_end"]), frame))
    return max(scored, key=lambda item: item[0])[1], modified


def collect_suvi(t0: pd.Timestamp, deadline: Deadline, log: list[str], workdir: Path | None = None):
    """-> (path of the NetCDF, frame record) or (None, None)."""
    workdir = Path(workdir or tempfile.mkdtemp(prefix="suvi-"))
    top = t0.floor("h")
    for back in range(MAX_BACK):
        slot = top - back * HOUR
        try:
            frame, modified = select_for_bin(slot, t0, deadline)
            if frame is None:
                log.append(f"suvi bin {slot:%F %H}Z: no frame published by t0")
                continue
            body = get(f"{S3_BASE}/{frame['key'][:-5]}.nc", deadline, timeout=120.0)
        except (RuntimeError, TimeoutError, ValueError, ET.ParseError) as error:
            log.append(f"suvi bin {slot:%F %H}Z: FAILED {error}")
            if deadline.left() <= 0:
                break
            continue
        if body is None:
            log.append(f"suvi bin {slot:%F %H}Z: NetCDF twin missing")
            continue
        path = workdir / frame["key"].rsplit("/", 1)[1].replace(".fits", ".nc")
        path.write_bytes(body)
        record = dict(bin=slot.isoformat(), obs_end=frame["obs_end"].isoformat(), key=frame["key"],
                      last_modified=modified[frame["key"]].isoformat())
        log.append(f"suvi bin {slot:%F %H}Z: frame ending {frame['obs_end']:%F %T}Z, {len(body) / 1e6:.1f} MB")
        return str(path), record
    return None, None

### 1.5 DONKI CME notifications

T·K 특징 8개의 원자료입니다. 유효한 예보는 발행 후 7일까지, 지난 도착 예보는 96시간 전까지만 쓰므로 30일치면 충분합니다(5단계에서 전체 기록과 같은 값이 나오는지 확인합니다).

In [ ]:
DONKI_API = "https://ccmc.gsfc.nasa.gov/DONKI-API/get/notifications?startDate={}&endDate={}&type=CME"
DONKI_LOOKBACK = pd.Timedelta(days=30)


def published_by(messages, t0):
    """API 발행 시각이 t0 이전인 메시지만, 메시지 ID마다 하나씩."""
    byid = {}
    for row in messages:
        issued = pd.to_datetime(row.get("messageIssueTime"), utc=True, errors="coerce")
        if not pd.isna(issued) and issued <= t0:
            byid[row.get("messageID")] = row
    return list(byid.values())


def collect_donki(t0, deadline, log):
    start, end = (t0 - DONKI_LOOKBACK).strftime("%Y-%m-%d"), t0.strftime("%Y-%m-%d")
    try:
        payload = get(DONKI_API.format(start, end), deadline, timeout=60.0)
    except (RuntimeError, TimeoutError) as error:
        log.append(f"donki cme: FAILED {error}")
        return None
    rows = json.loads(payload) if payload and payload.strip() else []
    if not isinstance(rows, list):
        log.append("donki cme: unexpected response")
        return None
    kept = published_by(rows, t0)
    log.append(f"donki cme: {len(rows)} notifications {start}..{end}, {len(kept)} issued by t0")
    return kept

### 1.6 수집 실행

In [ ]:
def collect(t0, workdir, budget=420.0):
    """원천 하나가 실패하면 그 항목만 비워 둡니다. 비어 있는 입력은 전처리에서 결측으로 처리됩니다."""
    deadline = Deadline(budget)
    bundle = SimpleNamespace(t0=t0, ace={}, ace_1m={}, enlil=[], suvi_path=None, suvi_frame=None, donki=None, log=[])
    bundle.ace = collect_ace(t0, deadline, bundle.log)
    try:                                    # 없으면 최신 ACE 칸은 forward fill로 남음
        bundle.ace_1m = collect_ace_1m(t0, deadline, bundle.log)
    except Exception as error:
        bundle.log.append(f"ace_1m: FAILED {type(error).__name__}: {error}")
    bundle.enlil = collect_enlil(t0, deadline, bundle.log)
    bundle.donki = collect_donki(t0, deadline, bundle.log)
    try:
        bundle.suvi_path, bundle.suvi_frame = collect_suvi(t0, deadline, bundle.log, workdir)
    except Exception as error:              # SUVI가 없으면 해당 열은 결측으로 booster에 들어감
        bundle.log.append(f"suvi: FAILED {type(error).__name__}: {error}")
    return bundle


began = time.perf_counter()
bundle = collect(T0, RUN_DIR)
(RUN_DIR / "bundle.json").write_text(json.dumps(dict(
    t0=T0.isoformat(), ace=bundle.ace, ace_1m=bundle.ace_1m, enlil=bundle.enlil, donki=bundle.donki,
    suvi_file=Path(bundle.suvi_path).name if bundle.suvi_path else None, suvi_frame=bundle.suvi_frame, log=bundle.log)))
print(f"collect: {time.perf_counter() - began:.1f}s")
print(*bundle.log, sep="\n")

## 2. 전처리

`t0`를 정시로 내린 시각이 origin O입니다. 학습 표와 같은 식으로 O 앞 168개 origin의 특징을 다시 만들고 forward fill한 뒤 O의 행만 씁니다.

### 2.1 ACE 특징

- 월별 파일 행(라벨 L)은 칸 L + 1h에 들어갑니다. status가 0이 아니거나 결측 표시값이면 NaN입니다.
- `<변수>_last`(칸 O), `_m6`, `_m24`(칸 평균), `v_lag{0,24,48,72}`(칸 O−k의 속도), `rec26/27/28`(target 시각의 26·27·28일 전 속도)과 그 평균 `rec_mean`.
- `t0`에는 방금 끝난 1시간(라벨 O−1h)이 월별 파일에 아직 없습니다. 그 칸은 실시간 1분/5분 목록에 이미 올라온 status 0 표본의 평균(E1)으로 채우고, 표본이 없으면 마지막 유효 월별 값을 씁니다.

In [ ]:
# product -> (fields after the 6 time fields: (column | marker, status marker), missing value per column)
ACE_SPEC = {
    "swepam": ([("S", None), ("ace_density_cm3", "S"), ("ace_speed_kms", "S"), ("ace_temperature_k", "S")],
               {"ace_density_cm3": -9999.9, "ace_speed_kms": -9999.9, "ace_temperature_k": -1.0e5}),
    "mag": ([("S", None), ("ace_bx_gsm_nt", "S"), ("ace_by_gsm_nt", "S"), ("ace_bz_gsm_nt", "S"), ("ace_bt_nt", "S"),
             ("ace_lat_gsm_deg", "S"), ("ace_lon_gsm_deg", "S")], -999.9),
    "epam": ([("Se", None), ("ace_epam_e_channel1", "Se"), ("ace_epam_e_channel2", "Se"), ("Sp", None),
              ("ace_epam_p_channel1", "Sp"), ("ace_epam_p_channel2", "Sp"), ("ace_epam_p_channel3", "Sp"),
              ("ace_epam_p_channel4", "Sp"), ("ace_epam_p_channel5", "Sp"), ("ace_epam_anis_index", "Sp")],
             {"ace_epam_anis_index": -1.0, "*": -1.0e5}),
    "sis": ([("S10", None), ("ace_sis_gt10_flux", "S10"), ("S30", None), ("ace_sis_gt30_flux", "S30")], -1.0e5),
}


def read_product(product: str, texts: list[str]) -> pd.DataFrame:
    """Monthly file texts -> clean values indexed by label time (status != 0 or a missing marker -> NaN)."""
    fields, missing = ACE_SPEC[product]
    names = [n for n, _ in fields]
    rows = []
    for order, text in enumerate(texts):
        for line in text.splitlines():
            if not line or line[0] in ":#":
                continue
            parts = line.split()
            if len(parts) != 6 + len(fields):
                raise ValueError(f"ace {product}: {len(parts)} fields in {line!r}")
            rows.append([order] + parts[:4] + parts[6:])
    if not rows:
        return pd.DataFrame()
    raw = pd.DataFrame(rows, columns=["file", "yr", "mo", "da", "hhmm"] + names)
    hhmm = raw["hhmm"].astype(int)
    t = pd.to_datetime(dict(year=raw.yr.astype(int), month=raw.mo.astype(int), day=raw.da.astype(int)), utc=True)
    label = t + pd.to_timedelta(hhmm // 100, unit="h") + pd.to_timedelta(hhmm % 100, unit="m")
    vals = raw[names].astype(float)
    vals.insert(0, "label_utc", label)
    vals.insert(1, "file", raw["file"])
    vals = vals.sort_values(["label_utc", "file"]).drop_duplicates("label_utc", keep="last").set_index("label_utc")
    clean = pd.DataFrame(index=vals.index)
    for name, status in fields:
        if not name.startswith("ace_"):
            continue
        v = vals[name].copy()
        miss = missing.get(name, missing.get("*")) if isinstance(missing, dict) else missing
        bad = np.isclose(v, miss, rtol=0, atol=1e-6) | ~np.isfinite(v)
        if status is not None:
            bad |= vals[status].ne(0)
        clean[name] = v.where(~bad)
    return clean


# model column -> (product, offset minutes, length minutes) of the hour's samples that the monthly hourly mean uses
# (research/ace_1m_to_hourly_20261108 stage 1, `identification.json` best_rule; 5-minute lists for EPAM and SIS)
ONE_MINUTE_RULE = {
    "ace_speed_kms": ("swepam", 0, 60), "ace_density_cm3": ("swepam", 1, 58), "ace_temperature_k": ("swepam", 0, 60),
    "ace_bt_nt": ("mag", 39, 2), "ace_by_gsm_nt": ("mag", 0, 60),
    "ace_epam_p_channel3": ("epam", 0, 60), "ace_sis_gt10_flux": ("sis", 0, 60),
}


def read_list(product: str, text: str) -> pd.DataFrame:
    """A real-time 1m/5m list (monthly row layout) -> clean values by sample stamp; malformed lines are skipped."""
    width = 6 + len(ACE_SPEC[product][0])
    good = [line for line in text.splitlines() if line and line[0] not in ":#" and len(line.split()) == width
            and line.split()[0].isdigit()]
    return read_product(product, ["\n".join(good) + "\n"]) if good else pd.DataFrame()


def newest_bin_mean(ace_1m: dict[str, str], origin: pd.Timestamp) -> tuple[dict[str, float], str]:
    """E1: plain mean of the status-0 samples of the just-closed hour (label origin - 1h) that are on the list at t0.

    Each column uses the sample window of its monthly rule; a column with no usable sample is left out (the caller keeps
    its forward fill). No model, no fit: this is the arithmetic the monthly file does, over the part of the hour that
    has arrived.
    """
    label = origin - HOUR
    values: dict[str, float] = {}
    counts: dict[str, int] = {}
    cache: dict[str, pd.DataFrame] = {}
    for column, (product, offset, length) in ONE_MINUTE_RULE.items():
        if product not in cache:
            cache[product] = read_list(product, ace_1m.get(product, ""))
        frame = cache[product]
        if column not in frame:
            continue
        stamp = frame.index
        window = frame[column][(stamp >= label + pd.Timedelta(minutes=offset)) & (stamp < label + pd.Timedelta(minutes=offset + length))].dropna()
        counts[column] = len(window)
        if len(window):
            values[column] = float(window.mean())
    note = (f"ace 1m E1 for bin {origin:%F %H}Z (label {label:%H}Z): {len(values)}/{len(ONE_MINUTE_RULE)} columns from "
            f"the real-time lists (samples used " + ", ".join(f"{c.removeprefix('ace_')}={counts.get(c, 0)}" for c in ONE_MINUTE_RULE) + ")")
    return values, note


def hourly_bins(ace: dict[str, list[str]], origin: pd.Timestamp, ace_1m: dict[str, str] | None = None) -> tuple[pd.DataFrame, str]:
    """All products on one hourly grid of bins (label + 1h) up to the origin bin.

    Bins after the newest published row (the hour that ended minutes ago and is not in the monthly file yet) are
    filled forward from the newest published bin, as the user decided on 2026-11-07. With `ace_1m` the origin bin of
    each model column is then replaced by the plain mean of the arrived 1-minute/5-minute samples of that hour (E1,
    `newest_bin_mean`); a column without a usable sample takes E0, the last valid monthly value (carried through NaN bins), and earlier bins are untouched. Interior
    gaps stay NaN here and are handled by the feature-level forward fill, as in training.
    """
    parts = [read_product(p, ace.get(p, [])) for p in ACE_SPEC]
    parts = [p for p in parts if len(p)]
    if not parts:
        return pd.DataFrame(), "no ACE rows"
    label = pd.concat(parts, axis=1).sort_index()
    label = label[~label.index.duplicated(keep="last")]
    label.index = bin_hour(label.index)
    grid = pd.date_range(label.index.min(), origin, freq="h")
    table = label.reindex(grid)
    newest = label.index.max()
    note = f"newest published ACE bin {newest:%F %H}Z"
    if newest < origin:
        trailing = grid[grid > newest]
        table.loc[trailing] = table.loc[newest].to_numpy()
        note += f"; {len(trailing)} unpublished trailing bin(s) filled forward"
        if ace_1m:
            means, one_minute_note = newest_bin_mean(ace_1m, origin)
            for column in ONE_MINUTE_RULE:
                if column not in table.columns:
                    continue
                if column in means:
                    table.loc[origin, column] = means[column]
                else:                                   # no usable sample: E0 = last valid monthly value, carried through NaN bins
                    earlier = table[column].iloc[:-1].dropna()
                    table.loc[origin, column] = earlier.iloc[-1] if len(earlier) else np.nan
            note += "; " + one_minute_note
    return table, note


def features_at(table: pd.DataFrame, origins: pd.DatetimeIndex) -> pd.DataFrame:
    """The notebook's `_last/_m6/_m24` and `v_lag*` for the needed variables, one row per origin (not yet filled)."""
    f = {}
    for c in table.columns:
        x = table[c]
        f[f"{c}_last"] = x
        f[f"{c}_m6"] = x.rolling(6, min_periods=1).mean()
        f[f"{c}_m24"] = x.rolling(24, min_periods=1).mean()
    speed = table["ace_speed_kms"]
    v = {f"v_lag{k}": speed.shift(k) for k in (0, 24, 48, 72)}
    return pd.concat([pd.DataFrame(v), pd.DataFrame(f)], axis=1).reindex(origins)


def recurrence(table: pd.DataFrame, origins: pd.DatetimeIndex) -> dict[str, np.ndarray]:
    """rec26/27/28 and rec_mean: ACE speed bin at (origin + h) - 26/27/28 days, shape (origins, 72), not yet filled."""
    speed = table["ace_speed_kms"]
    h = np.arange(1, 73)
    out = {}
    stack = []
    for d in (26, 27, 28):
        target = origins.to_numpy("datetime64[ns]")[:, None] + h.astype("timedelta64[h]").astype("timedelta64[ns]")[None, :]
        shifted = pd.DatetimeIndex(target.ravel(), tz="UTC") - pd.Timedelta(days=d)
        values = speed.reindex(shifted).to_numpy(float).reshape(len(origins), 72)
        out[f"rec{d}"] = values
        stack.append(values)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)        # all three days missing -> NaN, as in the notebook
        out["rec_mean"] = np.nanmean(np.stack(stack, 0), 0)
    return out

### 2.2 SUVI 특징

프레임 하나에서 어두운 영역(코로나홀) 면적 `suvi_t035_*`, `suvi_t055_*`와 위도 가운데 띠의 경도 셀 12개 통계를 구합니다. `strip_v{속도}_*`는 그 속도의 태양풍이 target 시각 O+h에 L1에 닿으려면 떠났어야 할 경도 주변 셀의 가중 평균이라서 lead마다 값이 다릅니다. `img_age_h`는 칸 시각 − 관측 끝(시간)입니다.

In [ ]:
THRESHOLDS = (0.35, 0.55)
OMEGA, D_KM, CELL_W, STRIP_HALF = 0.55, 147858620.7, 15.0, 5.0
SPEEDS = (350, 450, 550, 650, 750)
TAU = np.array([D_KM / v / 3600.0 for v in SPEEDS])
STRIP_STATS = {"logmed": "log_median", "dark": "dark_fraction", "cover": "valid_fraction"}
EDGES = -90.0 + CELL_W * np.arange(13)
HOURS = np.arange(1, 73)


def read_frame(path):
    with h5py.File(path, "r") as nc:
        var = nc["RAD"]
        raw = np.asarray(var[...])
        rad = raw.astype(np.float64)
        invalid = ~np.isfinite(rad)
        for name in ("_FillValue", "missing_value"):
            if name in var.attrs:
                invalid |= raw == np.asarray(var.attrs[name]).reshape(-1)[0]
        rad = rad * float(np.asarray(var.attrs.get("scale_factor", 1)).item())
        rad += float(np.asarray(var.attrs.get("add_offset", 0)).item())
        rad[invalid] = np.nan
        header = dict(nc.attrs)
        for key in ("CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "CDELT1", "CDELT2", "PC1_1", "PC1_2", "PC2_1", "PC2_2"):
            if key in nc:
                header[key] = np.asarray(nc[key][...]).item()
        return rad.squeeze(), np.asarray(nc["DQF"][...]).squeeze(), header


def lat1_cells(rad, dqf, header):
    """Statistics of the 12 longitude cells of latitude band 1 (the notebook's `suvi_lonII_lat1_*`)."""
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    h = {str(k).upper(): v for k, v in header.items()}

    def number(key, default=None):
        value = h.get(key, default)
        if value is None:
            raise ValueError(f"missing SUVI geometry: {key}")
        return float(np.asarray(value).item())

    yy, xx = np.indices(rad.shape, dtype=float)
    xx -= number("CRPIX1") - 1
    yy -= number("CRPIX2") - 1
    theta = np.deg2rad(number("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[number("PC1_1"), number("PC1_2")], [number("PC2_1"), number("PC2_2")]])
    rsun = number("RSUN")
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    x = (matrix[0, 0] * xx + matrix[0, 1] * yy) / rsun
    y = (matrix[1, 0] * xx + matrix[1, 1] * yy) / rsun
    disk = x * x + y * y <= 0.98 ** 2
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    z = np.sqrt(np.maximum(0, 1 - x * x - y * y))
    b0 = np.deg2rad(number("SOLAR_B0", 0))
    lat = np.arcsin(np.clip(y * np.cos(b0) + z * np.sin(b0), -1, 1))
    lon = np.arctan2(x, z * np.cos(b0) - y * np.sin(b0))
    longitude = np.clip(np.floor((lon + np.pi / 2) / np.pi * 12), 0, 11).astype(int)
    latitude = np.clip(np.floor((lat + np.pi / 2) / np.pi * 3), 0, 2).astype(int)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    result = {}
    for i in range(12):
        region = disk & (longitude == i) & (latitude == 1)
        values = rad[valid & region]
        log = np.log1p(values)
        result[f"suvi_lon{i:02d}_lat1_log_median"] = float(np.median(log)) if len(log) else np.nan
        result[f"suvi_lon{i:02d}_lat1_dark_fraction"] = float(np.mean(values < .5 * reference)) if len(log) else np.nan
        result[f"suvi_lon{i:02d}_lat1_valid_fraction"] = float(len(values) / region.sum()) if region.any() else 0.
    return result


def dark_geometry(shape, h):
    rsun = h["RSUN"]
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    y, x = np.indices(shape, dtype=float)
    x -= h["CRPIX1"] - 1
    y -= h["CRPIX2"] - 1
    theta = np.deg2rad(h.get("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]])
    xx = (matrix[0, 0] * x + matrix[0, 1] * y) / rsun
    yy = (matrix[1, 0] * x + matrix[1, 1] * y) / rsun
    disk = xx ** 2 + yy ** 2 <= .98 ** 2
    z = np.sqrt(np.maximum(1e-8, 1 - xx ** 2 - yy ** 2))
    b0 = np.deg2rad(h.get("SOLAR_B0", 0))
    lat = np.rad2deg(np.arcsin(np.clip(yy * np.cos(b0) + z * np.sin(b0), -1, 1)))
    lon = np.rad2deg(np.arctan2(xx, z * np.cos(b0) - yy * np.sin(b0)))
    area = np.where(disk, 1 / (z * 2 * np.pi * rsun ** 2), 0)
    return disk, lat, lon, area


def dark_regions(rad, dqf, coordinates, threshold):
    disk, lat, lon, area = coordinates
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    dark = valid & (rad < threshold * reference)
    structure = np.ones((3, 3), dtype=bool)
    dark = ndimage.binary_closing(ndimage.binary_opening(dark, structure), structure) & valid
    labels, count = ndimage.label(dark, structure)
    totals = np.bincount(labels.ravel(), weights=area.ravel(), minlength=count + 1)
    keep = totals >= .0005
    keep[0] = False
    mask = keep[labels]
    labels, count = ndimage.label(mask, structure)
    return mask, labels, count, valid, lat, lon, area


def dark_area_stats(rad, dqf, header):
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    keys = {"CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "PC1_1", "PC1_2", "PC2_1", "PC2_2"}
    h = {str(k).upper(): float(np.asarray(v).item()) for k, v in header.items() if str(k).upper() in keys}
    rsun = h["RSUN"]
    if all(key in h for key in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        scale = np.linalg.svd(np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]]), compute_uv=False).min()
        if scale <= 0:
            raise ValueError("singular SUVI PC matrix")
    else:
        scale = 1.
    radius = rsun / scale + 2
    cx, cy = h["CRPIX1"] - 1, h["CRPIX2"] - 1
    x0, x1 = max(0, int(np.floor(cx - radius))), min(rad.shape[1], int(np.ceil(cx + radius)) + 1)
    y0, y1 = max(0, int(np.floor(cy - radius))), min(rad.shape[0], int(np.ceil(cy + radius)) + 1)
    rad, dqf = rad[y0:y1, x0:x1], dqf[y0:y1, x0:x1]
    h["CRPIX1"], h["CRPIX2"] = cx + 1 - x0, cy + 1 - y0
    coordinates = dark_geometry(rad.shape, h)
    out = {}
    for threshold in THRESHOLDS:
        mask, labels, count, valid, lat, lon, area = dark_regions(rad, dqf, coordinates, threshold)
        result = {"area": float(area[mask].sum()), "count": float(count),
                  "valid_fraction": float(valid.sum() / max(1, (area > 0).sum()))}
        objects = []
        for label, bounds in enumerate(ndimage.find_objects(labels), 1):
            if bounds is None:
                continue
            inside = labels[bounds] == label
            weights = area[bounds][inside]
            la, lo = lat[bounds][inside], lon[bounds][inside]
            objects.append((float(weights.sum()), float(np.average(la, weights=weights)),
                            float(np.average(lo, weights=weights)), float(np.ptp(lo)), float(np.ptp(la))))
        objects.sort(key=lambda row: (-row[0], row[1], row[2]))
        result["max_area"] = objects[0][0] if objects else 0.
        for i in range(3):
            values = objects[i] if i < len(objects) else (0.,) * 5
            result.update({f"top{i + 1}_{name}": value for name, value in
                           zip(("area", "lat", "lon", "lon_width", "lat_width"), values)})
        for width in (15, 30):
            result[f"meridian{width}_area"] = float(area[mask & (np.abs(lon) <= width)].sum())
            result[f"meridian{width}_equatorial_area"] = float(area[mask & (np.abs(lon) <= width) & (np.abs(lat) <= 30)].sum())
        result["equatorial_area"] = float(area[mask & (np.abs(lat) <= 30)].sum())
        if not valid.any():
            result = {key: np.nan for key in result}
        out.update({f"suvi_t{round(threshold * 100):03d}_{key}": value for key, value in result.items()})
    return out


def frame_features(path) -> dict[str, float]:
    rad, dqf, header = read_frame(path)
    return {**lat1_cells(rad, dqf, header), **dark_area_stats(rad, dqf, header)}


def strip_columns(cells: dict[str, float], age_h: float) -> dict[str, np.ndarray]:
    """strip_v{350..750}_{logmed,dark,cover}: 72 values each, notebook cell 10 for a single origin."""
    lat1 = {s: np.array([cells[f"suvi_lon{i:02d}_lat1_{suffix}"] for i in range(12)]) for s, suffix in STRIP_STATS.items()}
    lam = OMEGA * (TAU[None, :] - (HOURS[:, None] + age_h))                                  # (72, 5)
    w = np.clip(np.minimum(lam[..., None] + STRIP_HALF, EDGES[1:]) - np.maximum(lam[..., None] - STRIP_HALF, EDGES[:-1]),
                0, None) / (2 * STRIP_HALF)                                                    # (72, 5, 12)
    ws = w.sum(-1)
    out = {}
    for s in STRIP_STATS:
        num = (w * lat1[s][None, None, :]).sum(-1)
        value = np.where(ws > 0, num / np.where(ws > 0, ws, 1), np.nan)                        # (72, 5)
        for k, speed in enumerate(SPEEDS):
            out[f"strip_v{speed}_{s}"] = value[:, k]
    return out


def image_age_h(bin_time: pd.Timestamp, obs_end: pd.Timestamp) -> float:
    return float((bin_time - obs_end) / pd.Timedelta(hours=1))

### 2.3 ENLIL 특징

완료 칸이 [O−7일, O]인 WSA-ENLIL 실행 중 도착 예상 시각이 있는 것만 씁니다. `enlil_n_pending`은 아직 지나지 않은(−6h 이후) 도착 예보 수, `enlil_kp`는 그 예보들의 최대 Kp입니다.

In [ ]:
WEEK = pd.Timedelta(days=7)


def enlil_at(records: list[dict], origin: pd.Timestamp) -> dict[str, float]:
    runs = []
    for s in records:
        if s.get("estimatedShockArrivalTime"):
            runs.append((pd.Timestamp(s["modelCompletionTime"]), pd.Timestamp(s["estimatedShockArrivalTime"]),
                         max(s.get("kp_90") or 0, s.get("kp_135") or 0, s.get("kp_180") or 0), bool(s.get("isEarthGB"))))
    if not runs:
        return dict(enlil_next_arr_h=np.nan, enlil_since_arr_h=np.nan, enlil_n_pending=0.0, enlil_kp=0.0,
                    enlil_glancing=np.nan)
    frame = pd.DataFrame(runs, columns=["done", "arrival", "kp", "glancing"]).drop_duplicates()
    frame["bin"] = bin_hour(frame["done"])
    window = frame[(frame["bin"] <= origin) & (frame["bin"] >= origin - WEEK)]
    dt = ((window["arrival"] - origin) / pd.Timedelta(hours=1)).to_numpy(float)
    fut, past = dt >= -6, (dt < -6) & (dt >= -96)
    kp, glancing = window["kp"].to_numpy(float), window["glancing"].to_numpy(bool)
    return dict(
        enlil_next_arr_h=dt[fut].min() if fut.any() else np.nan,
        enlil_since_arr_h=-dt[past].max() if past.any() else np.nan,
        enlil_n_pending=float(fut.sum()),
        enlil_kp=float(kp[fut].max()) if fut.any() else 0.0,
        enlil_glancing=float(glancing[fut].all()) if fut.any() else np.nan)

### 2.4 DONKI 도착 시각(T)·CME 운동학(K) 특징

이 모델이 기준선 S-cap에 더한 8개입니다. CME notification 본문에서 "NASA missions near Earth" 도달 예보를 읽습니다.

| 특징 | 뜻 |
|---|---|
| `pub_enlil_next_arr_h` | 아직 유효한 다음 지구 도달 예보까지 남은 시간 (h) |
| `pub_enlil_since_arr_h` | 6–96 h 전에 지난 가장 최근 도달 예보 이후 경과 시간 (h) |
| `pub_enlil_glancing` | 다음 예보가 glancing blow인지 (1 / 0 / −1) |
| `pub_cme_next_speed_kms`, `_half_angle_deg`, `_lon_deg`, `_lat_deg` | 다음 예보의 지구 대상 CME 중 가장 빠른 것의 속도·반각·방향 |
| `pub_cme_last_speed_kms` | 지난 예보 CME의 속도 |

- 본문 발행 시각이 origin 이전인 메시지만 씁니다. 같은 CME의 갱신·철회가 나오면 예전 예보를 지우고, 발행 후 7일이 지난 예보는 버립니다.
- −999는 "값 없음"입니다. 파싱 규칙은 2025-11-01 이전 메시지로 정한 뒤 고정한 것입니다.

In [ ]:
MISSING = -999.

TIME = r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}(?::\d{2})?Z'
CME_ID = r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}-CME-\d+'
MESSAGE_ID = r'\d{8}-AL-\d+'
EARTH = r'NASA missions near Earth'
ARRIVAL = re.compile(
    rf'(?P<modal>(?:will|may|might)\s+|(?:is|are)\s+estimated\s+to\s+)'
    rf'(?:possibly\s+)?(?:reach|impact)\s+{EARTH}\s+at\s+(?:about\s+)?(?P<arrival>{TIME})', re.I)
NEGATIVE = re.compile(
    rf'(?:will\s+not|is\s+not\s+expected\s+to|are\s+not\s+expected\s+to|'
    rf'is\s+no\s+longer\s+expected\s+to|are\s+no\s+longer\s+expected\s+to)'
    rf'\s+(?:reach|impact|affect)\s+{EARTH}', re.I)
KINEMATICS = re.compile(
    r'Estimated speed:\s*~(\d+) km/s\.\s*'
    r'Estimated opening half-angle:\s*(\d+) deg\.\s*'
    r'Direction \(lon\./lat\.\):\s*(-?\d+)/(-?\d+) in Heliocentric Earth Equatorial coordinates\.\s*'
    rf'Activity ID:\s*({CME_ID})')


def parse_message(message):
    """Return a traceable forecast/retraction or an explicit rejection record."""
    body = message.get('messageBody', '')
    record = dict(message_id=message.get('messageID'), message_url=message.get('messageURL'),
                  api_issue=message.get('messageIssueTime'), status='rejected', reason='',
                  body_issue=None, event_ids=[], previous_ids=[], arrival=None, glancing=-1,
                  earth_text='', action=None)
    issue_match = re.search(r'^##\s*Message Issue Date:\s*(\S+)', body, re.M | re.I)
    id_match = re.search(r'^##\s*Message ID:\s*(\S+)', body, re.M | re.I)
    if not issue_match or not id_match:
        record['reason'] = 'missing_body_identity_or_issue'
        return record
    api_issue = pd.to_datetime(record['api_issue'], utc=True, errors='coerce')
    issue = pd.to_datetime(issue_match.group(1), utc=True, errors='coerce')
    record['body_issue'] = None if pd.isna(issue) else issue.isoformat()
    if id_match.group(1) != record['message_id']:
        record['reason'] = 'body_message_id_mismatch'
        return record
    if pd.isna(issue) or pd.isna(api_issue) or not (api_issue <= issue < api_issue + pd.Timedelta(minutes=1)):
        record['reason'] = 'body_issue_not_within_api_issue_minute'
        return record
    section = re.split(r'##\s*Summary\s*:', body, maxsplit=1, flags=re.I)
    if len(section) != 2:
        record['reason'] = 'missing_summary'
        return record
    summary = re.split(r'##\s*Notes\s*:', section[1], maxsplit=1, flags=re.I)[0]
    if re.search(r'\bensemble\b', body[:body.find('## Summary:')] + summary, re.I):
        record['reason'] = 'ensemble_excluded'
        return record
    summary = re.split(r'Links? to (?:the )?(?:movies|simulations|modeled|modelled)', summary,
                       maxsplit=1, flags=re.I)[0]
    flat = re.sub(r'\s+', ' ', summary).strip()
    record['previous_ids'] = sorted(set(re.findall(MESSAGE_ID, flat)))
    all_ids = sorted(set(re.findall(CME_ID, flat)))
    paragraphs = [re.sub(r'\s+', ' ', p).strip() for p in re.split(r'\n\s*\n', summary)]
    positive = [(p, m) for p in paragraphs for m in ARRIVAL.finditer(p)]
    negative = [p for p in paragraphs if NEGATIVE.search(p)]
    if positive and negative:
        record['reason'] = 'conflicting_earth_actions'
        return record
    if negative:
        earth_text = ' '.join(negative)
        action, arrival, glancing = 'retract', None, -1
    elif positive:
        arrival_values = {m.group('arrival') for _, m in positive}
        if len(arrival_values) != 1:
            record['reason'] = 'multiple_earth_arrivals_ambiguous'
            record['earth_text'] = ' '.join(p for p, _ in positive)
            return record
        earth_text = ' '.join(dict.fromkeys(p for p, _ in positive))
        action, arrival = 'forecast', pd.Timestamp(next(iter(arrival_values)))
        if arrival <= issue:
            record['reason'] = 'arrival_not_after_public_issue'
            record['earth_text'] = earth_text
            return record
        geometries = set()
        for paragraph, match in positive:
            prefix = paragraph[:match.start()]
            # The nearest geometry must directly qualify the CME reaching Earth.
            direct = re.search(r'(leading edge|flank)\s+of\s+(?:the\s+)?CMEs?\s*$', prefix, re.I)
            if direct:
                geometries.add(0 if direct.group(1).lower() == 'leading edge' else 1)
            if re.search(rf'glancing blow\s+(?:at|to)\s+{EARTH}', paragraph, re.I):
                geometries.add(1)
        glancing = next(iter(geometries)) if len(geometries) == 1 else -1
    else:
        record['reason'] = 'no_explicit_deterministic_earth_arrival_or_retraction'
        return record
    record['earth_text'] = earth_text
    earth_ids = sorted(set(re.findall(CME_ID, earth_text)))
    if earth_ids:
        event_ids = earth_ids
    elif len(all_ids) == 1:
        event_ids = all_ids
    elif len(all_ids) > 1 and re.search(r'\bCMEs\b', earth_text):
        # The publication explicitly describes the listed CMEs as a joint Earth forecast.
        event_ids = all_ids
    elif len(all_ids) > 1:
        dates = set(re.findall(TIME, earth_text))
        event_ids = [identifier for identifier in all_ids
                     if identifier.split('-CME-')[0][:16] + 'Z' in dates]
        if len(event_ids) != 1:
            record['reason'] = 'ambiguous_earth_event_identity'
            return record
    else:
        # A previous-message-only relation is resolved chronologically by parse_corpus.
        event_ids = []
        if not record['previous_ids']:
            record['reason'] = 'missing_earth_event_identity'
            return record
    record.update(status='accepted', reason='explicit_public_earth_' + action,
                  event_ids=event_ids, action=action,
                  arrival=None if arrival is None else arrival.isoformat(), glancing=glancing)
    return record


def parse_corpus(messages):
    parsed = [parse_message(m) for m in messages]
    parsed.sort(key=lambda r: (r['body_issue'] or '', r['message_id']))
    earlier = {}
    for record in parsed:
        if record['status'] != 'accepted':
            continue
        if not record['event_ids']:
            previous = [earlier[k] for k in record['previous_ids'] if k in earlier]
            groups = {tuple(r['event_ids']) for r in previous}
            if len(groups) != 1 or len(previous) != len(record['previous_ids']):
                record.update(status='rejected', reason='ambiguous_or_unavailable_previous_identity')
                continue
            record['event_ids'] = list(next(iter(groups)))
        earlier[record['message_id']] = record
    return parsed


def kinematics(body, event_ids):
    """Fastest published CME among the Earth-forecast events of one notification, or None."""
    parts = re.split(r'##\s*Summary\s*:', body, maxsplit=1, flags=re.I)
    if len(parts) != 2:
        return None
    summary = re.split(r'##\s*Notes\s*:', parts[1], maxsplit=1, flags=re.I)[0]
    blocks = [dict(speed=float(a), half_angle=float(b), lon=float(c), lat=float(d), cme=e)
              for a, b, c, d, e in KINEMATICS.findall(summary)]
    if (not blocks or len(blocks) != len(re.findall(r'Estimated speed', summary))
            or len({b['cme'] for b in blocks}) != len(blocks)):
        return None
    chosen = [b for b in blocks if b['cme'] in set(event_ids)] or blocks
    return max(chosen, key=lambda b: (b['speed'], b['cme']))


def features_at_origins(messages, origins):
    """T and K at each sorted unique origin; only messages issued at or before the origin count."""
    origins = pd.DatetimeIndex(pd.to_datetime(origins, utc=True))
    if not origins.is_monotonic_increasing or origins.has_duplicates:
        raise ValueError('origins must be sorted and unique')
    bodies = {m['messageID']: m.get('messageBody', '') for m in messages}
    records = [r for r in parse_corpus(messages) if r['status'] == 'accepted']
    records.sort(key=lambda r: (pd.Timestamp(r['body_issue']), r['message_id']))
    speed = {r['message_id']: kinematics(bodies[r['message_id']], r['event_ids'])
             for r in records if r['action'] == 'forecast'}
    out = np.full((len(origins), len(DONKI_FEATURES)), MISSING, np.float32)
    out[:, 2] = -1.
    active, position = {}, 0
    for i, origin in enumerate(origins):
        while position < len(records) and pd.Timestamp(records[position]['body_issue']) <= origin:
            new = records[position]
            new_ids = set(new['event_ids'])
            for key, old in list(active.items()):
                if new_ids.intersection(old['event_ids']):
                    del active[key]       # an update or retraction of the same CME replaces the old forecast
            if new['action'] == 'forecast':
                active[new['message_id']] = new
            position += 1
        for key, old in list(active.items()):
            if pd.Timestamp(old['body_issue']) < origin - pd.Timedelta(days=7):
                del active[key]
        values = [((pd.Timestamp(r['arrival']) - origin) / pd.Timedelta(hours=1), r) for r in active.values()]
        pending = sorted((x for x in values if x[0] >= -6), key=lambda x: (x[0], x[1]['message_id']))
        past = sorted((x for x in values if -96 <= x[0] < -6), key=lambda x: (-x[0], x[1]['message_id']))
        if pending:
            hours, record = pending[0]
            out[i, 0], out[i, 2] = hours, record['glancing']
            cme = speed[record['message_id']]
            if cme:
                out[i, 3:7] = cme['speed'], cme['half_angle'], cme['lon'], cme['lat']
        if past:
            hours, record = past[0]
            out[i, 1] = -hours
            cme = speed[record['message_id']]
            if cme:
                out[i, 7] = cme['speed']
    frame = pd.DataFrame(out, columns=DONKI_FEATURES)
    frame.insert(0, 'origin', origins)
    return frame

### 2.5 조립

In [ ]:
def preprocess(bundle):
    """수집 결과 -> origin 하나의 모델 입력: 상수 31개(CONSTANT 순서)와 lead별 (72, 8) 배열(VARYING 순서)."""
    origin = bundle.t0.floor("h")
    origins = pd.date_range(origin - (LOOKBACK_ORIGINS - 1) * HOUR, origin, freq="h")
    notes, constant, varying, last_speed = [], {}, {}, None

    table, note = hourly_bins(bundle.ace, origin, bundle.ace_1m)                 # ACE
    notes.append(note)
    if len(table):
        constant.update(features_at(table, origins).ffill().iloc[-1].to_dict())
        for name, values in recurrence(table, origins).items():
            varying[name] = pd.DataFrame(values).ffill().iloc[-1].to_numpy(float)
        speed = table["ace_speed_kms"].dropna()
        last_speed = float(speed.iloc[-1]) if len(speed) else None

    if bundle.suvi_path and bundle.suvi_frame:                                   # SUVI
        cells = frame_features(bundle.suvi_path)
        age = image_age_h(pd.Timestamp(bundle.suvi_frame["bin"]), pd.Timestamp(bundle.suvi_frame["obs_end"]))
        constant.update({k: v for k, v in cells.items() if k.startswith("suvi_t")})
        constant["img_age_h"] = age
        varying.update(strip_columns(cells, age))
        notes.append(f"suvi frame of bin {bundle.suvi_frame['bin']} (age {age:.3f} h)")
    else:
        notes.append("no SUVI frame: suvi_*, img_age_h and strip_* are missing")

    constant.update(enlil_at(bundle.enlil, origin))                              # ENLIL

    if bundle.donki is not None:                                                 # DONKI T, K
        row = features_at_origins(bundle.donki, [origin]).iloc[0]
        constant.update({name: float(row[name]) for name in DONKI_FEATURES})
    else:
        notes.append("no DONKI CME notifications: pub_* are missing")

    c = np.array([constant.get(name, np.nan) for name in CONSTANT], np.float32)
    e = np.stack([np.asarray(varying.get(name, np.full(H, np.nan)), float) for name in VARYING], 1).astype(np.float32)
    missing = [n for n, v in zip(CONSTANT, c) if not np.isfinite(v)] + [
        n for k, n in enumerate(VARYING) if not np.isfinite(e[:, k]).all()]
    if missing:
        notes.append("missing after fill: " + ", ".join(missing))
    return SimpleNamespace(t0=bundle.t0, origin=origin, constant=c, varying=e, last_speed_kms=last_speed, notes=notes)


features = preprocess(bundle)
(RUN_DIR / "features.json").write_text(json.dumps(dict(
    t0=T0.isoformat(), origin=features.origin.isoformat(),
    constant={n: (float(v) if np.isfinite(v) else None) for n, v in zip(CONSTANT, features.constant)},
    varying={n: [float(v) if np.isfinite(v) else None for v in features.varying[:, k]] for k, n in enumerate(VARYING)},
    last_speed_kms=features.last_speed_kms, notes=features.notes)))
print(*features.notes, sep="\n")
display(pd.Series(features.constant, index=CONSTANT, name=f"origin {features.origin:%F %H}Z").to_frame())
display(pd.DataFrame(features.varying, index=pd.RangeIndex(1, H + 1, name="h"), columns=VARYING).iloc[[0, 23, 47, 71]])

## 3. 추론

체크포인트 하나(`s{seed}.npz`)에는 블록 3개(lead 1–24, 25–48, 49–72)마다 A900 booster `full_b{i}`와 S-cap continuation stage 6개 `full_b{i}_stage{k}`, stage별 수락 계수 `alphas`, 학습 구간에서만 추정해 고정한 lead별 offset(`calibration`, 창 2026-07-28 ~ 10-28)이 들어 있습니다(`prior_*`는 offset을 추정할 때 쓴 모델이라 추론에는 쓰지 않습니다).

- A900 booster는 smooth-tail MSE(lead별 학습 q80 위쪽 target에 최대 4배 가중)로 900 round 학습했고, continuation은 spectral 항 없이 MSE 상한 1.01 안에서 direct-S를 낮추는 stage만 받아들였습니다. 추론 식은 기존과 같습니다.
- 블록 예측 = A900 예측 + Σ alpha × stage 예측
- seed 예보 = clip(clip(블록 예측, 1, 3000) + offset, 1, 3000)

In [ ]:
def load_booster(buffer, threads):
    booster = xgb.Booster(params={"nthread": threads, "device": "cpu"})
    booster.load_model(bytearray(np.asarray(buffer, np.uint8).tobytes()))
    return booster


def load_checkpoint(path, threads=THREADS):
    """full 모델의 블록별 (A900 booster, [(alpha, stage booster)])와 lead별 offset."""
    with np.load(path, allow_pickle=False) as arrays:
        alphas = json.loads(arrays["alphas"].tobytes().decode())
        state = json.loads(arrays["calibration"].tobytes().decode())
        blocks = []
        for i in range(H // BLOCK):
            name = f"full_b{i}"
            stages = [(alpha, load_booster(arrays[f"{name}_stage{k}"], threads))
                      for k, alpha in enumerate(alphas[name]) if alpha]
            blocks.append((load_booster(arrays[name], threads), stages))
    offset = np.asarray(state["offset"], np.float64)
    if offset.shape != (H,):
        raise ValueError("offset must hold one value per lead")
    return SimpleNamespace(blocks=blocks, offset=offset, state=state)


def design(constant, varying, start):
    """열 순서: 부모 상수 20, 블록의 24 lead x 8, 추가 상수 11. constant (n, 31), varying (n, 72, 8)."""
    base = len(PARENT_CONSTANT)
    return np.hstack([constant[:, :base], varying[:, start:start + BLOCK].reshape(len(constant), -1),
                      constant[:, base:]]).astype(np.float32)


def predict_seed(model, constant, varying):
    raw = np.empty((len(constant), H), np.float64)
    for (reference, stages), start in zip(model.blocks, range(0, H, BLOCK)):
        x = design(constant, varying, start)
        block = reference.inplace_predict(x).astype(np.float64).reshape(len(x), -1)
        for alpha, stage in stages:
            block += alpha * stage.inplace_predict(x).astype(np.float64).reshape(block.shape)
        raw[:, start:start + BLOCK] = block
    corrected = np.clip(np.clip(raw, 1., 3000.) + model.offset[None, :], 1., 3000.)
    if not np.isfinite(corrected).all():
        raise ValueError("Nonfinite final forecasts")
    return corrected


MODELS = {seed: load_checkpoint(path) for seed, path in CHECKPOINTS.items()}
for seed, model in MODELS.items():
    print(f"s{seed}: stages used per block {[len(stages) for _, stages in model.blocks]}, "
          f"offset {model.offset.min():+.2f}..{model.offset.max():+.2f} km/s, "
          f"calibration {model.state['calibration_start'][:10]}..{model.state['calibration_end'][:10]}")

began = time.perf_counter()
by_seed = {seed: predict_seed(model, features.constant[None, :], features.varying[None, :, :])[0]
           for seed, model in MODELS.items()}
print(f"inference: {time.perf_counter() - began:.2f}s")

## 3.5 seam 평활화 (xf8)

보드 행 v2 `experiment.py`의 `smooth`를 그대로 옮겼습니다. 이음매 b(24, 48)마다 국소 기울기 `slope`를 넘는 계단 `jump`를 구해, 이음매 앞 8개 lead에는 더하고 뒤 8개 lead에서는 빼되 이음매에서 `jump/2`로 시작해 선형으로 줄입니다. 보정량은 원래 곡선에서 계산하고, 결과는 [1, 3000]으로 자릅니다.

seed 예보마다 적용한 뒤 세 seed를 등가중 평균합니다(보드 행과 같은 순서).

In [ ]:
SEAMS = (24, 48)


def smooth(pred, width=SEAM_TAPER):
    """pred: (n, 72). xf{width} seam correction computed on the original curve, then clip [1, 3000]."""
    p = np.asarray(pred, np.float64)
    q = p.copy()
    for b in SEAMS:
        slope = .5 * ((p[:, b - 1] - p[:, b - 2]) + (p[:, b + 1] - p[:, b]))
        jump = (p[:, b] - p[:, b - 1]) - slope
        for i in range(width):
            taper = .5 * jump * (width - i) / width
            q[:, b - 1 - i] += taper
            q[:, b + i] -= taper
    return np.clip(q, 1., 3000.)


smoothed = {seed: smooth(by_seed[seed][None, :])[0] for seed in SEEDS}
raw_forecast = np.mean([smoothed[seed] for seed in SEEDS], axis=0)
before = np.mean([by_seed[seed] for seed in SEEDS], axis=0)
print(f"seam xf{SEAM_TAPER}: " + ", ".join(
    f"h{b}|{b + 1} step {before[b] - before[b - 1]:+.2f} -> {raw_forecast[b] - raw_forecast[b - 1]:+.2f} km/s" for b in SEAMS))

## 4. 예보 기록

유한한 값 72개인지 확인하고 `forecast.json`을 씁니다(`forecast_speed_kms[0]`은 `t0 + 1h`의 값). 모델 출력이 쓸 수 없으면 마지막 속도 관측으로 채운 persistence를 대신 내고 `source`에 이유를 남깁니다.

In [ ]:
def finalize(t0, values, last_speed_kms):
    values = None if values is None else np.asarray(values, float)
    if values is not None and values.shape == (H,) and np.isfinite(values).all():
        return np.clip(values, 1.0, 3000.0), "model"
    known = last_speed_kms is not None and np.isfinite(last_speed_kms) and last_speed_kms > 0
    return np.full(H, last_speed_kms if known else FALLBACK_KMS), "persistence" + ("" if known else " (constant)")


values, source = finalize(T0, raw_forecast, features.last_speed_kms)
payload = {"t0": T0.strftime("%Y-%m-%dT%H:%M:%SZ"), "valid_from": (T0 + HOUR).strftime("%Y-%m-%dT%H:%M:%SZ"),
           "forecast_speed_kms": [round(float(v), 2) for v in values]}
assert len(payload["forecast_speed_kms"]) == H
(RUN_DIR / "forecast.json").write_text(json.dumps(payload, separators=(",", ":")))
(RUN_DIR / "forecast_by_seed.json").write_text(json.dumps(
    dict(t0=payload["t0"], source=source, period=PERIOD, seam_variant=f"xf{SEAM_TAPER}",
         seeds={str(s): smoothed[s].tolist() for s in SEEDS}, seeds_before_seam={str(s): by_seed[s].tolist() for s in SEEDS})))

print("source          :", source)
print("last ACE speed  :", features.last_speed_kms, "km/s")
print("forecast.json   :", RUN_DIR / "forecast.json")
spread = np.ptp([smoothed[s] for s in SEEDS], axis=0)
print(f"seed spread     : mean {spread.mean():.2f}, max {spread.max():.2f} km/s")
display(pd.DataFrame({"valid_utc": [T0 + k * HOUR for k in range(1, H + 1)], "forecast_kms": np.round(values, 2),
                      **{f"s{s}": np.round(smoothed[s], 2) for s in SEEDS}},
                     index=pd.RangeIndex(1, H + 1, name="h")).iloc[[0, 5, 11, 23, 35, 47, 59, 71]])

## 5. 검증

이 노트북의 추론·평활화 코드와 DONKI 특징 코드가 학습 쪽과 같은 계산인지 확인합니다. 예보 값에는 영향을 주지 않습니다. 학습 로더와 같이 lead별로 origin 순서대로 forward fill한 특징을 씁니다.

1. **보드 행 재현:** `data-v2/input/df.parquet`의 F3 query origin 31개 특징과 보드 실행 때 저장한 DONKI 메시지로 만든 8개 특징을, 보드 F3 체크포인트(`research/e2000fs_..._smooth_tail_nospec/weights/F3`)로 `predict_seed`한 뒤 3.5의 `smooth`(xf8)를 적용합니다. 보드 seam 행(v2) `s42/s43/s44/mean3_predictions.parquet`의 F3 구간과 값이 완전히 같아야 합니다. F3 구간은 날짜가 아니라 기간별 query 수(F1 2089, F2 2113, F3 2113)로 자릅니다.
2. **새 체크포인트 재현:** `~/sunrun-seam-live/input/df_live.parquet`의 probe origin 72개(2026-10-29T00Z ~ 10-31T23Z)를 이 노트북의 `MODELS`로 예측한 값이 `train_live.py`가 학습 직후 저장한 probe 예측과 완전히 같아야 합니다.
3. **30일 창:** 1.5의 30일치 메시지만으로 만든 DONKI 특징이 전체 기록(2022-01-01~)으로 만든 것과 같아야 합니다.

In [ ]:
DF, DF_LIVE = REPO / "data-v2/input/df.parquet", LIVE / "input/df_live.parquet"
RESULT = REPO / "research/result" / EXPERIMENT
BASE_CONSTANT = PARENT_CONSTANT + ADDED_CONSTANT
QUERY_COUNTS = {"F1": 2089, "F2": 2113, "F3": 2113}


def model_inputs(path, period, messages):
    """Query origins of one period: forward fill by lead over sorted origin (training loader rule) + DONKI features."""
    table = pd.read_parquet(path, columns=["set", "period", "origin", "h", *BASE_CONSTANT, *VARYING])
    table = table.sort_values(["h", "origin"], kind="stable")
    table[BASE_CONSTANT + VARYING] = table.groupby("h", sort=False)[BASE_CONSTANT + VARYING].ffill()
    table = table[table["set"].eq("query") & table["period"].eq(period)].sort_values(["origin", "h"])
    origins = pd.DatetimeIndex(pd.to_datetime(table["origin"].iloc[::H], utc=True))
    assert len(table) == len(origins) * H and origins.is_unique
    constant = table.loc[table["h"].eq(1).to_numpy(), BASE_CONSTANT].to_numpy(np.float32)
    varying = table[VARYING].to_numpy(np.float32).reshape(len(origins), H, len(VARYING))
    donki = features_at_origins(messages, origins)[DONKI_FEATURES].to_numpy(np.float32)
    return origins, np.hstack([constant, donki]), varying


# 1. board row, F3
board_corpus = json.loads((REPO / "research/result" / BASE_ROW / "logs/donki_notifications.json").read_text())
origins, C, E = model_inputs(DF, "F3", board_corpus)
assert len(origins) == QUERY_COUNTS["F3"]
start = H * (QUERY_COUNTS["F1"] + QUERY_COUNTS["F2"])                    # F3 rows of the stacked F1/F2/F3 frame
keys = np.repeat(origins.strftime("%Y-%m-%dT%H:%M:%SZ"), H)
check = {}
for seed in SEEDS:
    model = load_checkpoint(REPO / "research" / BASE_ROW / "weights/F3" / f"s{seed}.npz")
    check[seed] = smooth(predict_seed(model, C, E))
    board = pd.read_parquet(RESULT / f"results/s{seed}_predictions.parquet").iloc[start:start + H * len(origins)]
    assert np.array_equal(board["origin_last_input_utc"].to_numpy(), keys)
    assert np.array_equal(board["horizon_hours"].to_numpy(), np.tile(np.arange(1, H + 1), len(origins)))
    difference = np.abs(check[seed].ravel() - board["pred_kms"].to_numpy())
    assert difference.max() == 0., f"s{seed}: max diff {difference.max()}"
    print(f"board s{seed}: F3 {len(origins)} origins x {H} leads (xf{SEAM_TAPER}) identical to the seam row")
board = pd.read_parquet(RESULT / "results/mean3_predictions.parquet").iloc[start:start + H * len(origins)]
mean3 = np.stack([check[seed].ravel() for seed in SEEDS]).mean(axis=0)
assert np.array_equal(board["origin_last_input_utc"].to_numpy(), keys) and np.abs(mean3 - board["pred_kms"].to_numpy()).max() == 0.
print("board mean3: identical to the seam row")

# 2. new checkpoints, probe origins of df_live
live_corpus = json.loads((LIVE / "input/donki_notifications.json").read_text())
probe, PC, PE = model_inputs(DF_LIVE, PERIOD, live_corpus)
for seed, model in MODELS.items():
    saved = pd.read_parquet(LIVE / f"runs/seam_live_s{seed}/results/s{seed}_probe_predictions.parquet")
    assert np.array_equal(saved["origin_last_input_utc"].to_numpy(), np.repeat(probe.strftime("%Y-%m-%dT%H:%M:%SZ"), H))
    difference = np.abs(predict_seed(model, PC, PE).ravel() - saved["pred_kms"].to_numpy())
    assert difference.max() == 0., f"live s{seed}: max diff {difference.max()}"
    print(f"live s{seed}: {len(probe)} probe origins ({probe.min():%F %H}Z..{probe.max():%F %H}Z) identical to train_live.py")

# 3. DONKI 30-day window (board F3 origins)
corpus = board_corpus
sample = origins[:: max(1, len(origins) // 150)]
full = features_at_origins(corpus, sample)[DONKI_FEATURES].to_numpy()
window = np.vstack([features_at_origins(
    [m for m in published_by(corpus, o) if pd.to_datetime(m["messageIssueTime"], utc=True) >= (o - DONKI_LOOKBACK).floor("D")],
    [o])[DONKI_FEATURES].to_numpy() for o in sample])
assert np.array_equal(full, window)
print(f"DONKI 30-day window: 8 features identical to the full record at {len(sample)} origins "
      f"({int((full[:, 0] != MISSING).sum())} with a pending arrival forecast)")